# Tutorial

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

SMALL_SIZE = 16
MEDIUM_SIZE = 20
BIGGER_SIZE = 24

# plt.rc("font", **{"family": "sans-serif", "sans-serif": ["Helvetica"]})
# plt.rc("text", usetex=True)
plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

%matplotlib inline
# %matplotlib widget

To illustrate the `CARDS` library, consider a simpler linear regression problem based on the following likelihood

$$ \mathbf{y} \mid \mathbf{x} \sim \mathcal{N}( \mathbf{Hx},\sigma^2 \mathbf{I}_{M}). $$

### Data generation

In [2]:
from cards.utils.utils_observations import compute_sigma2_from_isnr
from cards.core.execution_context import ExecutionContext
from cards.io.io_manager import IOManager
from cards.logger import build_logger
from pathlib import Path
import cards.backend as xp
from os.path import exists
from os import makedirs

# defining execution context
ctx = ExecutionContext("serial", "cpu")

# IOManager used later on (saving/loading results and data)
io_mng = IOManager(ctx)

# FIXME: to modify to properly create paths
# logger
# log_path = Path("logs")
# if not exists(log_path):
#     makedirs(log_path)
logger = build_logger(ctx.rank)

# problem dimensions
obs_path = "data.h5"
# if not exists(obs_path):
#     makedirs(obs_path)
dimension = 100  # size of unknown parameter x (ground truth)
sample_size = 50  # size of observations y
isnr = 35  # observation iSNR (in dB)
seed_data = 3

In [3]:
def generate_data(
    sample_size: int,
    dimension: int,
    isnr: float,
    seed: int,
    intercept: float = 0.0,
    ctx: ExecutionContext | None = None,
):
    r"""Generate synthetic data with controlled ground truth for a sparse linear
    regression problem.

    Parameters
    ----------
    sample_size : int
        Number of observed data samples.
    dimension : int
        Dimension of each sample.
    isnr: float
        The desired input SNR (in dB).
    seed : int
        Seed to initialize the random number generator used to generate the
        data.
    intercept : float, optional
        Value of the intercept vector :math:`\mu`, by default 0.
    ctx: ExecutionContext, optional
        Execution context object containing MPI properties (e.g., is_mpi, comm, rank).

    Returns
    -------
    H : xp.ndarray
        Matrix whose rows are the data samples.
    y : xp.ndarray
        Observation to be analyzed.
    x : xp.ndarray
        Ground thruth value for the regression vector :math:`\beta`.
    sigma2 : float
        Ground truth value for the variance of the white Gaussian noise.
    support : xp.ndarray[bool]
        Array of boolean indicating the support of the ground truth vector
        :math:`\beta`.
    """
    rng = xp.random.default_rng(seed)

    # set up parameters
    # noise_std = 1
    proportion_of_nonzero_coefficients = 0.1
    signal_absolute_value = 10

    # prepare containers to return
    x = rng.normal(0.0, 0.25, size=(dimension,))
    support = rng.choice(
        2,
        size=(dimension,),
        p=[
            1 - proportion_of_nonzero_coefficients,
            proportion_of_nonzero_coefficients,
        ],
    )
    id_support = xp.flatnonzero(support)
    positive_peaks_position = rng.choice(2, size=id_support.shape, p=[0.5, 0.5])

    x[id_support] = -signal_absolute_value + rng.standard_normal(size=id_support.shape)
    x[id_support[positive_peaks_position.astype(bool)]] += 2 * signal_absolute_value

    H = rng.normal(0, 1, (sample_size, dimension))
    Hx = H @ x + intercept
    sigma2 = compute_sigma2_from_isnr(Hx, isnr, ctx)
    y = rng.normal(Hx, xp.sqrt(sigma2))

    return y, H, x, sigma2, xp.where(support)[0]

In [4]:
# helper functions (forcing serial run and bypassing usage of hooks for simplicity)
from dataclasses import dataclass


@dataclass
class LassoObs:
    y: xp.ndarray
    x: xp.ndarray
    H: xp.ndarray
    sigma2: float
    isnr: float
    seed_data: int
    comm_size: int
    is_gpu: bool


def save_observations(
    io_mng: IOManager,
    obs: LassoObs,
    obs_path: Path,
) -> None:
    with io_mng.open_master_only(obs_path, mode="x") as f:
        if f is not None:
            obs_dict = {
                "sigma2": obs.sigma2,
                "isnr": obs.isnr,
                "seed_data": obs.seed_data,
                "comm_size": obs.comm_size,
                "is_gpu": obs.is_gpu,
            }
            io_mng.write_config(f, obs_dict)
            io_mng.write_array(f, "y", obs.y)
            io_mng.write_array(f, "x", obs.x)
            io_mng.write_array(f, "H", obs.H)


def load_observations(
    ctx: ExecutionContext,
    io_mng: IOManager,
    obs_path: Path,
) -> LassoObs:
    with io_mng.open(obs_path, mode="r", force_serial=True) as f:
        y = io_mng.read_array(f, "y")
        x = io_mng.read_array(f, "x")
        H = io_mng.read_array(f, "H")
        obs_dict = io_mng.read_config(f)

    return LassoObs(
        y,
        x,
        H,
        obs_dict["sigma2"],
        obs_dict["isnr"],
        obs_dict["seed_data"],
        ctx.comm_size,
        ctx.is_gpu,
    )

In [6]:
y, H, x, sigma2, indices_support = generate_data(
    sample_size, dimension, isnr, seed=seed_data, ctx=ctx
)

obs = LassoObs(y, x, H, sigma2, isnr, seed_data, ctx.comm_size, ctx.is_gpu)

save_observations(
    io_mng,
    obs,
    obs_path,
)

## First approach

Considering a standard Lasso regularization leads to a target posterior distribution with density

$$ p(\mathbf{x} \mid \mathbf{y}) \propto \exp \Big(-\frac{1}{2 \sigma^2} \| \mathbf{y} - \mathbf{H}\mathbf{x} \|_2^2 -\lambda \| \mathbf{x} \|_1 \Big) $$

The non-smooth potential function can be addressed with a Langevin-based MCMC algorithm to generate a Markov chain, whose invariant distribution approximates this target.

To draw samples from this probability law using `CARDS`, the user must define its own model, inheriting from a base model (`cards.base_model.BaseModel` or `cards.base_model.BaseDistributedModel`), or define the virtual methods inherited from the mother class.


### Model definition

In [16]:
from abc import abstractmethod
import cards.backend as xp
from cards.core.variable import Variable
from cards.models.base_model import BaseModel, BaseDistributedModel
from cards.transition_kernels.base_transition_kernel import BaseTransitionKernel
from cards.operators.linear_operator import LinearOperator


class MatrixOperator(LinearOperator):
    r"""Linear-operator interface added on top of a ndarray for compatibility with generic CARDS objects.

    Attributes
    ----------
    matrix : xp.ndarray
        Matrix representation of the linear operator, assumed real-valued.

    Raises
    ------
    ValueError
        matrix should a tensor of order 2 (len(matrix) == 2)
    """

    def __init__(self, matrix: xp.ndarray):
        super().__init__((matrix.shape[1],), (matrix.shape[0],))
        if not len(matrix.shape) == 2:
            raise ValueError("matrix should a tensor of order 2 (len(matrix) == 2)")
        self.matrix = matrix

    def forward(self, image: xp.ndarray, op=None) -> xp.ndarray:
        return self.matrix @ image

    def adjoint(self, data: xp.ndarray, adjoint_op=None) -> xp.ndarray:
        return self.matrix.T @ data


@dataclass
class LassoParams:
    r"""Configuration parameters for Losso problems under additive white Gaussian noise.

    This data class encapsulates the known variables and hyperparameters required to
    define the forward model of this inverse problem.
    """

    sigma2: float
    r"""The variance of the additive white Gaussian noise, :math:`\sigma^2`."""

    reg_coeff: float
    r"""The regularization coefficient used to weight the prior relative to the likelihood."""


class BaseLassoModel(BaseModel):
    r"""Abstract base class for Lasso problem under additive white Gaussian noise.

    This class extends :class:`BaseModel` to provide a standard framework for
    Lasso tasks. It stores the forward operator, initializes internal
    buffers for convolution operations, and retains the model hyperparameters.

    Parameters
    ----------
    estimators : list[BaseEstimator]
        A list of estimator builders used to compute parameter estimates during sampling.
    params : LassoParams
        The configuration parameters containing the observations and the noise variance.
    measurement_operator : LinearOperator
        The measurement operator, assumed linear in the example considered.
    X : BaseTransitionKernel
        The transition kernel responsible for sampling the primary target variable.
    """

    def __init__(
        self,
        params: LassoParams,
        measurement_operator: MatrixOperator,
        y: Variable,
        X: BaseTransitionKernel,
        *other_kernels: BaseTransitionKernel,
    ):
        super().__init__(X.var, *[k.var for k in other_kernels])
        self.params = params

        self.reg_coeff = params.reg_coeff
        self.sigma2 = params.sigma2

        self.H = measurement_operator

        self.y = y
        self.X = X

        self.Hx = self.H.forward(self.X.state)

    @abstractmethod
    def set_conditionals(self) -> None:
        r"""Set up the conditional distributions for the transition kernels.

        This method is called automatically at the end of initialization.
        Inheriting classes must implement this to define how the transition
        kernels update their respective variables based on the specific
        inference algorithm (e.g., Gibbs sampling or PnP-ULA).
        """

    def compile(self) -> None:
        self.set_conditionals()

In [ ]:
from cards.core.variable import Variable
import torch
from cards.functionals.prox import prox_l1norm


class LassoModel(BaseLassoModel):
    def __init__(
        self,
        params: LassoParams,
        measurement_operator: MatrixOperator,
        y: Variable,
        X: BaseTransitionKernel,
    ):
        super().__init__(params, measurement_operator, y, X)
        self.reg_coeff = params.reg_coeff
        self.sigma2 = params.sigma2

    def set_conditionals(self) -> None:
        """Set the conditionals of the transition kernels including the coupling between those kernels."""
        if isinstance(self.X, PSGLA):
            self.X.prox = lambda state: prox_l1norm(
                state, self.X.step_size * self.reg_coeff
            )
            self.X.grad = lambda state: self.H.adjoint(self.Hx - y) / self.sigma2
        else:
            raise ValueError("Kernel type not yet supported by this model.")

    def _on_states_updated(self):
        self.Hx = self.H.forward(self.X.state)

    def update(self, rng: np.random.Generator | torch.Generator):
        """Gobal update of the model. Updates every kernel used by the model and computes annex variables.

        Parameters
        ----------
        rng : np.random.Generator | torch.Generator
            Random number generator, given by the sampler.
        """
        self.X.mc_step(rng)
        self.Hx = self.H.forward(self.X.state)

    def compute_potential(self) -> float:
        """compute_potential Computes the potential.

        Returns
        -------
        float
            Potential of the targeted law.
        """
        p = 0.5 / self.sigma2 * xp.sum((y - self.Hx) ** 2)
        p += self.reg_coeff * xp.sum(xp.abs(self.X.state))
        return p

### Drawing samples from the posterior distribution
#### Set parameters

In [13]:
from cards.transition_kernels.psgla import PSGLA
from cards.samplers.sampler import SamplerParameters
from os.path import exists
from os import makedirs
from cards.random import create_rng


nsamples = 5000
nb_ckpts = 5
ckpt_size = nsamples // nb_ckpts
ckpt_prefix = Path("data/psgla_bayesian_lasso")
seed = 1234

reg_coeff = 5e-2

# rng for the sampler
rng = create_rng(seed, ctx)

if not exists(ckpt_prefix):
    makedirs(ckpt_prefix)

sampler_params = SamplerParameters(ckpt_size, nb_ckpts, ckpt_prefix, "sample", seed)

#### Helper functions to define estimators and intantiate the model

In [14]:
from cards.transition_kernels.psgla import CpuPSGLA, GpuPSGLA
from cards.estimators.base_estimator import BaseEstimator
from cards.estimators.ci import CI
from cards.estimators.mmse_var import MMSEVar
from cards.core.layout import Layout


def build_estimators(
    obs: LassoObs,
) -> tuple[dict[str, Variable], list[BaseEstimator]]:

    # ! imposing serial layout (no sharding) for simplicity
    layout_y = Layout(obs.y.shape, obs.y.shape, None)
    y_var = Variable(
        layout=layout_y,
        name="Y",
        state=obs.y,
        dtype=obs.y.dtype,
    )

    layout_x = Layout((obs.H.shape[1],), (obs.H.shape[1],), None)
    x_var = Variable(
        layout=layout_x,
        name="X",
        dtype=obs.x.dtype,
    )

    variables = {"X": x_var, "Y": y_var}
    estimators: list[BaseEstimator] = [MMSEVar(x_var), CI(x_var, all_samples=True)]

    return variables, estimators


def build_model(
    ctx: ExecutionContext,
    reg_coeff,
    obs: LassoObs,
    vars_: dict[str, Variable],
) -> BaseModel:
    model_params = LassoParams(obs.sigma2, reg_coeff)
    step_size_X = 0.99 * obs.sigma2 / (xp.linalg.norm(obs.H, ord=2) ** 2)

    x_var = vars_["X"]
    y_var = vars_["Y"]

    PSGLA = GpuPSGLA if ctx.is_gpu else CpuPSGLA

    X = PSGLA(
        var=x_var,
        step_size=step_size_X,
    )

    if not ctx.is_mpi:
        Model = LassoModel
    else:
        raise ValueError(
            "Only serial implementation currently provided for the tutorial example."
        )

    return Model(
        model_params,
        MatrixOperator(obs.H),
        y=y_var,
        X=X,
    )

#### Build model and sampler

In [ ]:
from cards.samplers.sampler import Sampler

# build estimators and model
vars_, estimators = build_estimators(obs)
lasso_model = build_model(
    ctx,
    reg_coeff,
    obs,
    vars_,
)

# build sampler
sampler = Sampler.create_from_context(
    ctx, io_mng, lasso_model, estimators, sampler_params, logger
)

In [19]:
# run the sampler
sampler.sample()

Sampling |                                                                                                    | 0/5 [  0%]
2026-09-20 16:53:23,194 - Rank 0 - INFO     -   │    │ Checkpoint 1/5 | Potential:  4.090e+01 | t/it (s):  4.636e-05
Sampling |████████████████████                                                                                | 1/5 [ 20%] | ETA: 00m 00s
2026-09-20 16:53:23,292 - Rank 0 - INFO     -   │    │ Checkpoint 2/5 | Potential:  4.154e+01 | t/it (s):  4.530e-05
Sampling |████████████████████████████████████████                                                            | 2/5 [ 40%] | ETA: 00m 00s
2026-09-20 16:53:23,390 - Rank 0 - INFO     -   │    │ Checkpoint 3/5 | Potential:  3.945e+01 | t/it (s):  4.710e-05
Sampling |████████████████████████████████████████████████████████████                                        | 3/5 [ 60%] | ETA: 00m 00s
2026-09-20 16:53:23,488 - Rank 0 - INFO     -   │    │ Checkpoint 4/5 | Potential:  4.061e+01 | t/it (s):  4.649

### Post-processing : estimates and reconstruction metrics

In [ ]:
# FIXME: adapt tutorial example from here


def compute_snr(recons, truth):
    return 20 * np.log(np.linalg.norm(truth) / np.linalg.norm(truth - recons))

In [ ]:
# FIXME: to update
from os.path import join
import h5py
import matplotlib.pyplot as plt


def plot_results(path, data_thruth, sample_size, nb_batches):
    potential = np.zeros(sample_size)
    mmse = np.zeros_like(data_thruth)
    full_batch = np.zeros((sample_size, *data_thruth.shape))
    for i in range(1, nb_batches + 1):
        file_name = join(path, "sample" + str(i) + ".h5")
        with h5py.File(file_name) as file:
            mmse += file["MMSE"]

            potential[(i - 1) * batch_size : i * batch_size] = file["potential"][:]
            full_batch[(i - 1) * batch_size : i * batch_size, ...] = file["batch/beta"][
                :
            ]
    mmse = mmse / nb_batches
    plt.figure()
    plt.plot(mmse, label="estimation")
    plt.plot(beta_true, label="truth")
    plt.title("Reconstruction")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(np.abs(mmse - data_thruth))
    plt.title("Absolute value of the error")
    plt.show()

    plt.figure()
    plt.plot(potential)
    plt.title("Potential")
    plt.show()

    quantile = np.quantile(full_batch, (0.25, 0.75), axis=0)
    plt.figure()
    plt.plot(mmse[-15:], label="estimator")
    plt.plot(quantile[0, -15:], label="q_25", color="green")
    plt.plot(quantile[1, -15:], label="q_75", color="red")
    plt.legend()
    plt.title(r"Credible interval (last 15 values of $\beta$)")
    plt.show()

    plt.figure()
    autocorr = np.correlate(full_batch[:, 0], full_batch[:, 0], "same")
    plt.plot(autocorr[sample_size // 2 :] / np.amax(autocorr))
    plt.title("Autocorrelation")
    plt.show()

    print("SNR estimate/truth : ", compute_snr(mmse, data_thruth))

In [ ]:
# compute estimates


In [ ]:
# ! FIXME: to update
plot_results("data/psgla_bayesian_lasso", beta_true, nsamples, nb_batches)

# Second approach based on exact data augmentation

The problem can also equivalently reformulated using with an exact data augmentation trick, introducing an intermediary variable $\boldsymbol{\tau}$ in the hierachical model as follows ([Bayesian Lasso, Park and Casella 2008](https://people.eecs.berkeley.edu/~jordan/courses/260-spring09/other-readings/park-casella.pdf)):

\begin{align*}
    p(\mathbf{x} \mid \mathbf{y}, \boldsymbol{\tau}) &\propto \exp \Big(-\frac{1}{2\sigma^2} \| \mathbf{H}\mathbf{x} - \mathbf{y} \|_2^2 - \frac{1}{2} \| \mathbf{x} \|^2_{\sigma^2 \mathbf{D}_{\boldsymbol{\tau}}}- \frac{\lambda}{2} \| \boldsymbol{\tau} \|_2^2 \Big), \\
    \mathbf{x} \mid \boldsymbol{\tau} &\sim \mathcal{N}(0,\sigma^2 \mathbf{D}_{\boldsymbol{\tau}}), \\
    p(\boldsymbol{\tau}) &= \displaystyle{\prod_{j=1}^M} \Big( \dfrac{\lambda^2}{2}e^{-\lambda^2\tau_j^2} \Big),
\end{align*}

with $\mathbf{D}_{\boldsymbol{\tau}} = \mathrm{diag}(\boldsymbol{\tau})$.
The associated conditional distributions are given by

\begin{align*}
    (\forall j \in \{1, \dotsc, N\}) \quad \tau_j \mid \mathbf{y}, \mathbf{x} &\sim  \text{InvGaussian} \left( \sqrt{ \dfrac{\lambda^2\sigma^2}{x_j} }, \lambda^2\right), \\
    \mathbf{x} \mid \boldsymbol{\tau}, \mathbf{y} &\sim \mathcal{N} (\mathbf{S}, \mathbf{m}),
\end{align*}

with $\mathbf{S} = (\boldsymbol{\Phi}^T \boldsymbol{\Phi}+ \mathbf{D}_{\boldsymbol{\tau}})^{-1}$, $\boldsymbol{\Phi} = \dfrac{1}{\sigma} \mathbf{H}$, $\mathbf{m} = \mathbf{S}\boldsymbol{\Phi}^T \boldsymbol{\alpha}$ and $\boldsymbol{\alpha} = \dfrac{1}{\sigma} \mathbf{y}$. Note that samples can be efficciently drawn from $\mathbf{x} \mid \boldsymbol{\tau}, \mathbf{y}$ due to the specific structure of $\mathbf{S}$ (see [Bhattacharya2016](https://arxiv.org/abs/1506.04778)).

All the variables can be sampled directly and efficiently using a Gibbs sampler, implemented in the next sections.

## Implementation

In [ ]:
# TODO: see simpler writing for such a simple kernel (too much boilerplate for now)
# TODO: rewrite as a single kernel, with a trabsition involving two variables

# def update_x(tau2, Phi, alpha, rng):
#     u = xp.sqrt(tau2) * rng.normal(scale=1, size=tau2.shape)
#     nu = Phi @ u + rng.normal(size=Phi.shape[0])
#     w = xp.linalg.solve(
#         Phi @ (tau2[:, None] * Phi.T)
#         + xp.identity(Phi.shape[0]),
#         alpha - nu,
#     )
#     # linear solve more efficient with scipy, can use symmetry
#     return u + tau2 * (Phi.T @ w)


class AugmentedBLassoConditionalX(BaseTransitionKernel):
    def __init__(
        self,
        var: Variable,
    ) -> None:
        super().__init__(var)

    @abstractmethod
    def _noise(self, state: xp.ndarray, shape: tuple[int, ...], rng) -> xp.ndarray: ...

    def get_variables(self, state):
        raise ValueError("Phi, tau2, alpha not defined.")

    def mc_step(self, rng):
        Phi, tau2, alpha = self.get_variables(self.state)

        u = xp.sqrt(tau2) * self._noise(self.var.state, tau2.shape, rng)
        # rng.standard_normal(size=tau2.shape)
        nu = Phi @ u + self._noise(self.var.state, Phi.shape[0], rng)
        # rng.standard_normal(size=Phi.shape[0])
        w = xp.linalg.solve(
            Phi @ (tau2[:, None] * Phi.T) + xp.identity(Phi.shape[0]),
            alpha - nu,
        )
        # linear solve more efficient with scipy, can use symmetry
        return u + tau2 * (Phi.T @ w)


class CpuAugmentedBLassoConditionalX(AugmentedBLassoConditionalX):
    def _noise(
        self, state: xp.ndarray, shape: tuple[int, ...], rng: np.random.Generator
    ) -> xp.ndarray:
        return rng.standard_normal(size=shape, dtype=state.dtype)


class GpuAugmentedBLassoConditionalX(AugmentedBLassoConditionalX):
    def _noise(self, state, shape, rng: torch.Generator) -> xp.ndarray:
        return xp.asarray(
            torch.normal(
                mean=0.0,
                std=1.0,
                size=shape,
                generator=rng,
                device=rng.device,
            ),
            state.dtype,
        )


# def update_tau2(x, reg_coeff, sigma2, rng):
#     return 1.0 / rng.wald(
#         xp.sqrt(sigma2) * reg_coeff / xp.abs(x), reg_coeff**2
#     )


class AugmentedBLassoConditionalTau2(BaseTransitionKernel):
    def __init__(
        self,
        var: Variable,
    ) -> None:
        super().__init__(var)

    def get_variables(self, state):
        raise ValueError("x, reg_coeff, sigma2 not defined.")

    def mc_step(self, rng):
        x, reg_coeff, sigma2 = self.get_variables(self.state)

        # NOTE: no equivalent of invese Gaussian distribution in torch for now...
        return 1.0 / rng.wald(xp.sqrt(sigma2) * reg_coeff / xp.abs(x), reg_coeff**2)

In [ ]:
class AugmentedLassoModel(BaseLassoModel):
    def __init__(
        self,
        params: LassoParams,
        measurement_operator: MatrixOperator,
        y: Variable,
        X: AugmentedBLassoConditionalX,
        Tau2: AugmentedBLassoConditionalTau2,
    ):
        super().__init__(params, measurement_operator, y, X, Tau2)
        self.Tau2 = Tau2

        # internal cached variables to facilitate implementation
        self.Phi_ = self.H.matrix / xp.sqrt(self.sigma2)
        self.alpha_ = self.y.state / xp.sqrt(self.sigma2)

    def set_conditionals(self) -> None:
        """Set the conditionals of the transition kernels including the coupling between those kernels."""
        self.X.get_variables = lambda state: self.Phi_, self.Tau2.state, self.alpha_
        self.Tau2.get_variables = (
            lambda state: self.X.state,
            self.reg_coeff,
            self.sigma2,
        )

    def _on_states_updated(self):
        self.Hx = self.H.forward(self.X.state)

    def update(self, rng):
        self.X.mc_step(rng)
        self.Hx = self.H.forward(self.X.state)
        self.Tau2.mc_step(rng)

    def compute_potential(self):

        p = 0.5 / self.sigma2 * xp.sum((y - self.Hx) ** 2)
        p += 0.5 * xp.sum(1 / (self.sigma2 * self.Tau2.state) * self.X.state**2)
        p += 0.5 * self.reg_coeff * xp.sum(self.Tau2.state)
        return p

#### Helper functions to define estimators and intantiate the model

In [ ]:
def build_estimators(
    obs: LassoObs,
) -> tuple[dict[str, Variable], list[BaseEstimator]]:

    # ! imposing serial layout (no sharding) for simplicity
    layout_y = Layout(obs.y.shape, obs.y.shape, None)
    y_var = Variable(
        layout=layout_y,
        name="Y",
        state=obs.y,
        dtype=obs.y.dtype,
    )

    layout_x = Layout((obs.H.shape[1],), (obs.H.shape[1],), None)
    x_var = Variable(
        layout=layout_x,
        name="X",
        dtype=obs.x.dtype,
    )

    tau2_var = Variable(
        layout=layout_x,
        name="Tau2",
        dtype=obs.x.dtype,
    )

    variables = {"X": x_var, "Y": y_var, "Tau2": tau2_var}

    estimators: list[BaseEstimator] = [MMSEVar(x_var), CI(x_var, all_samples=True)]

    return variables, estimators


def build_model(
    ctx: ExecutionContext,
    reg_coeff,
    obs: LassoObs,
    vars_: dict[str, Variable],
) -> BaseModel:
    model_params = LassoParams(obs.sigma2, reg_coeff)

    x_var = vars_["X"]
    y_var = vars_["Y"]
    tau2_var = vars_["Tau2"]

    # NOTE: for now, Cpu implementation enforced (not counterpart of inverse Gaussian distribution in torch)
    # TODO: generalize implementation to GPU
    X = CpuAugmentedBLassoConditionalX(
        var=x_var,
    )

    Tau2 = AugmentedBLassoConditionalTau2(
        var=tau2_var,
    )

    if not ctx.is_mpi:
        Model = AugmentedLassoModel
    else:
        raise ValueError(
            "Only serial implementation currently provided for the tutorial example."
        )

    return Model(
        model_params,
        MatrixOperator(obs.H),
        y=y_var,
        X=X,
        Tau2=Tau2,
    )

## Drawing samples from the posterior distribution

#### Set parameters

In [ ]:
nsamples = 5000
nb_ckpts = 5
ckpt_size = nsamples // nb_ckpts
ckpt_prefix = Path("data/augmented_bayesian_lasso")
seed = 1234

reg_coeff = 2e1

rng = create_rng(seed, ctx)

if not exists(ckpt_prefix):
    makedirs(ckpt_prefix)

sampler_params = SamplerParameters(ckpt_size, nb_ckpts, ckpt_prefix, "sample", seed)

#### Instanciate model and sampler

In [30]:
from cards.samplers.sampler import Sampler

# build estimators and model
vars_, estimators = build_estimators(obs)
augmented_lasso_model = build_model(
    ctx,
    reg_coeff,
    obs,
    vars_,
)

# set an initial value for Tau2
augmented_lasso_model.Tau2.var.state = rng.exponential(
    2 / reg_coeff**2, size=augmented_lasso_model.Tau2.state.shape
)

# build sampler
sampler = Sampler.create_from_context(
    ctx, io_mng, lasso_model, estimators, sampler_params, logger
)

In [31]:
# run the sampler
sampler.sample()

Sampling |                                                                                                    | 0/5 [  0%]
2026-09-20 18:33:16,710 - Rank 0 - INFO     -   │    │ Checkpoint 1/5 | Potential:  4.581e+01 | t/it (s):  4.747e-05
Sampling |████████████████████                                                                                | 1/5 [ 20%] | ETA: 00m 00s
2026-09-20 18:33:16,811 - Rank 0 - INFO     -   │    │ Checkpoint 2/5 | Potential:  4.682e+01 | t/it (s):  5.168e-05
Sampling |████████████████████████████████████████                                                            | 2/5 [ 40%] | ETA: 00m 00s
2026-09-20 18:33:16,912 - Rank 0 - INFO     -   │    │ Checkpoint 3/5 | Potential:  4.500e+01 | t/it (s):  5.586e-05
Sampling |████████████████████████████████████████████████████████████                                        | 3/5 [ 60%] | ETA: 00m 00s
2026-09-20 18:33:17,012 - Rank 0 - INFO     -   │    │ Checkpoint 4/5 | Potential:  4.703e+01 | t/it (s):  4.773

## Analyze the generated data

In [ ]:
# compute estimates


In [ ]:
# FIXME: to update
plot_results("data/augmented_bayesian_lasso", beta_true, nsamples, nb_batches)

### References

1. [Bayesian Lasso, Park and Casella 2008](https://people.eecs.berkeley.edu/~jordan/courses/260-spring09/other-readings/park-casella.pdf)
2. [Bhattacharya2016](https://arxiv.org/abs/1506.04778)